# Vacation Recommendations

> **File:** `vacations.ipynb`  
> **Data:** `resources/cities_weather.csv`  
> **Purpose:** The application of a set of specified weather conditions to filter the global dataset down to a shortlist of candidate vacation destinations

---

## Executive Summary

This notebook is a companion to `weather.ipynb`. It takes the city weather dataset produced by that notebook — **714 cities** sampled globally via `citipy` and the OpenWeatherMap API — and narrows them down to locations that meet a specific set of ideal holiday conditions. For each qualifying city it then queries the Geoapify Places API to find a nearby hotel, restaurant, and tourist attraction, and plots everything on an interactive map.

### Notebook Structure

| Section | Content |
|---------|---------|
| 1 | Import city weather CSV; display all cities on a map |
| 2 | Apply weather filters; display qualifying cities |
| 3 | Query Geoapify for nearest hotel per city; map results |
| 4 | Query Geoapify for nearest restaurant per city; map results |
| 5 | Query Geoapify for nearest tourist attraction per city; final map |

### Ideal Weather Criteria

The filter applied in Section 2 is deliberately narrow:

| Condition | Range |
|-----------|-------|
| Temperature | 70–95 °F |
| Humidity | 35–65% |
| Cloudiness | 0–10% |
| Wind speed | 0–10 m/s |

These four filters applied in sequence reduce the 714-city dataset significantly — temperature alone cuts it to 286 cities, humidity to 55, and the near-clear-sky cloudiness requirement to 9. The wind speed filter leaves **3 qualifying cities**.

### Qualifying Cities

Based on the `cities_weather.csv` data collected on 18 March 2026, exactly three cities meet all conditions:

| City | Country | Temp (°F) | Humidity (%) | Cloudiness (%) | Wind (m/s) |
|------|---------|-----------|--------------|----------------|------------|
| Changem | India | 87.8 | 45 | 5 | 4.6 |
| Avenal | United States | 72.1 | 57 | 1 | 4.5 |
| Myeik | Myanmar | 86.9 | 60 | 5 | 8.4 |

The cloudiness threshold (≤ 10%) is the most restrictive filter by far: only 9 of 714 cities — just 1.3% of the sample — have near-clear skies at the time of data collection. This reflects the snapshot nature of the dataset; results will differ on every run.

### What the Notebook Produces

For each qualifying city the notebook calls the Geoapify Places API to locate the nearest hotel, restaurant, and tourist attraction, appending each to the city record. The final output is a series of interactive Folium maps — one per enrichment stage — with tooltips showing the venue name, city, and country. The final map (Figure 5.4) shows all three layers together.

---


In [1]:
 #*******************************************************************************************
 #
 #  File Name:  vacations.ipynb 
 #
 #  File Description:
 #      This Jupyter Notebook, vacations.ipynb, is a Python script to determine the 
 #      ideal locations (city and hotel) for a vacation and displays information on 
 #      a map.
 #      
 #
 #  Date                Description                                 Programmer
 #  ---------------     ------------------------------------        ------------------
 #  08/12/2023          Initial Development                         Nicholas J. George
 #  03/04/2026          Upgraded Module                             Nicholas J. George
 #
 #******************************************************************************************/

import citipyx
import mapx

import logx
import pandasx

import warnings

import pandas as pd

from bokeh.util.warnings import BokehUserWarning


warnings.filterwarnings('ignore')

warnings.simplefilter(action = 'ignore', category = BokehUserWarning)


pd.options.mode.chained_assignment = None

In [2]:
CONSTANT_LOCAL_FILE_NAME = 'vacations.ipynb'

In [3]:
logx.set_log_mode(False)

logx.set_image_mode(False)


logx.begin_program('vacations')

In [4]:
citipyx.set_vacation_temp_rng(70, 95)

citipyx.set_vacation_humid_rng(35, 65)

citipyx.set_vacation_cloud_rng(0, 10)

citipyx.set_vacation_wind_speed_rng(0, 10)

# <br> **Section 1: Vacation Data Acquisition**

## **1.1: Data Import from CSV File**

In [5]:
city_weather_df \
    = pd.read_csv \
        (citipyx.config_dict['data']['datafile'],
         index_col = citipyx.config_dict['params']['index'])

logx.log_write_obj(city_weather_df)

## **1.2: Display City Weather Data Set**

In [6]:
pandasx.rtn_fmt_tbl(city_weather_df, 'Table: 1.2: City Weather Information')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time
fish,63.75,-68.51,-22.31,67,100,0.00,CA,2026-03-18 23:58:01
stanley,54.87,-1.70,43.63,92,75,2.30,GB,2026-03-18 23:57:53
grytviken,-54.28,-36.51,44.73,75,7,21.74,GS,2026-03-18 23:58:02
quibala,-10.73,14.98,67.51,90,35,1.05,AO,2026-03-18 23:58:03
bilibino,68.05,166.44,-8.14,89,5,6.96,RU,2026-03-18 23:58:04
waitangi,-43.95,-176.56,60.82,92,100,1.99,NZ,2026-03-18 23:58:04
isafjordur,66.08,-23.12,44.26,91,100,27.40,IS,2026-03-18 23:58:05
steamboat springs,40.48,-106.83,43.36,39,0,3.44,US,2026-03-18 23:58:05
tura,25.52,90.22,84.36,45,100,6.06,IN,2026-03-18 23:58:06
yellowknife,62.46,-114.35,-11.52,82,40,3.44,CA,2026-03-18 23:55:02


## **1.3: Display City Weather Locations**

In [7]:
mapx.set_tooltip_display(False)

mapx.disp_folium_circles_df(city_weather_df, 'Figure 1.3: City Weather Locations')

# <br> **Section 2: Desired Weather Locations**

## **2.1: Establish Desired Weather Conditions for Vacation Locations**

In [8]:
weather_dict = citipyx.get_weather_dict()

vacations_df \
    = city_weather_df \
        .loc[(city_weather_df['temperature'] \
                >= weather_dict['min_temp']) \
             & (city_weather_df['temperature'] \
                <= weather_dict['max_temp']), :]

vacations_df \
    = vacations_df \
        .loc[(vacations_df['humidity'] \
                >= weather_dict['min_humid']) \
             & (vacations_df['humidity'] \
                <= weather_dict['max_humid']), :]

vacations_df \
    = vacations_df \
        .loc[(vacations_df['cloudiness'] \
                >= weather_dict['min_cloud']) \
             & (vacations_df['cloudiness'] \
                <= weather_dict['max_cloud']), :]

vacations_df \
    = vacations_df \
        .loc[(vacations_df['wind_speed'] \
                >= weather_dict['min_wind_speed']) 
             & (vacations_df['wind_speed'] \
                <= weather_dict['max_wind_speed']), :]

vacations_df.dropna(inplace = True)

vacations_df.reset_index(drop = True, inplace = True)


logx.log_write_obj(vacations_df)

## **2.2: Display Vacation Data Set**

In [9]:
pandasx.rtn_fmt_tbl(vacations_df, 'Table: 2.3: Vacation Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time
gundlupet,11.80,76.68,83.73,39,7,7.58,IN,2026-03-18 23:59:32
queenstown,-31.90,26.88,70.12,59,1,2.73,ZA,2026-03-18 23:59:45
bealana,-14.55,48.73,76.89,60,1,4.09,MG,2026-03-19 00:02:07
cape san lucas,22.89,-109.91,72.75,56,0,3.00,MX,2026-03-18 23:57:47
balabac,7.99,117.06,84.25,63,10,7.49,PH,2026-03-19 00:02:45
ontario,34.06,-117.65,72.90,42,0,3.44,US,2026-03-19 00:04:56
holy rosalia,27.32,-112.28,74.10,45,0,9.66,MX,2026-03-19 00:05:39


## **2.3: Display Vacation Locations**

In [10]:
mapx.set_tooltip_display(True)

mapx.disp_folium_circles_df(vacations_df, 'Figure 2.4: Vacation Locations')

# <br> **Section 3: Hotel Locations**

## **3.1: Add Hotel Column to DataFrame**

In [11]:
hotels_df = vacations_df.copy()

hotels_df['hotel_name'] = pd.Series(dtype = 'str')

hotels_df.reset_index(drop = True, inplace = True)

logx.log_write_obj(hotels_df)

## **3.2: Find Hotel Locations**

In [12]:
updated_hotels_df \
    = citipyx.update_location_vacation_df \
        (hotels_df, 
         'hotel_name', 
         'accommodation.hotel')

logx.log_write_obj(updated_hotels_df)

STARTING HOTEL SEARCH...


Located the following hotel...Khushi Resort in gundlupet, IN


Located the following hotel...Queens Hotel in gundlupet, IN


Located the following hotel...Comfort Rooms in gundlupet, IN


Located the following hotel...Travelodge Ontario in gundlupet, IN


Located the following hotel...Sun and Sea in gundlupet, IN


HOTEL SEARCH COMPLETE...




## **3.3: Display Hotel Data Set**

In [13]:
pandasx.rtn_fmt_tbl(updated_hotels_df, 'Table: 3.3: Hotel Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time,hotel_name
gundlupet,11.80,76.68,83.73,39,7,7.58,IN,2026-03-18 23:59:32,khushi resort
queenstown,-31.90,26.88,70.12,59,1,2.73,ZA,2026-03-18 23:59:45,queens hotel
cape san lucas,22.89,-109.91,72.75,56,0,3.00,MX,2026-03-18 23:57:47,comfort rooms
ontario,34.06,-117.65,72.90,42,0,3.44,US,2026-03-19 00:04:56,travelodge ontario
holy rosalia,27.32,-112.28,74.10,45,0,9.66,MX,2026-03-19 00:05:39,sun and sea


## **3.4: Display Hotel Locations**

In [14]:
mapx.set_tooltip_cols(['hotel_name', 'city', 'country'])

mapx.disp_folium_circles_df(updated_hotels_df, 'Figure 3.4: Hotel Locations')

# <br> **Section 4: Restaurant Locations**

## **4.1: Add Restaurant Column to DataFrame**

In [15]:
restaurant_df = updated_hotels_df.copy()

restaurant_df['restaurant_name'] = pd.Series(dtype = 'str')

restaurant_df.reset_index(drop = True, inplace = True)

logx.log_write_obj(restaurant_df)

## **4.2: Find Restaurant Locations**

In [16]:
updated_restaurant_df \
    = citipyx.update_location_vacation_df \
        (restaurant_df, 
         'restaurant_name', 
         'catering.restaurant')

logx.log_write_obj(updated_restaurant_df)

STARTING RESTAURANT SEARCH...


Located the following restaurant...Madena Food in gundlupet, IN


Located the following restaurant...N6 Road House in gundlupet, IN


Located the following restaurant...Taqueria los Paises I in gundlupet, IN


Located the following restaurant...Gloria's Mexican Cuisine in gundlupet, IN


Located the following restaurant...The Bolerian in gundlupet, IN


RESTAURANT SEARCH COMPLETE...




## **4.3: Display Restaurant Data Set**

In [17]:
pandasx.rtn_fmt_tbl(updated_restaurant_df, 'Table: 4.3: Restaurant Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time,hotel_name,restaurant_name
gundlupet,11.80,76.68,83.73,39,7,7.58,IN,2026-03-18 23:59:32,khushi resort,madena food
queenstown,-31.90,26.88,70.12,59,1,2.73,ZA,2026-03-18 23:59:45,queens hotel,n6 road house
cape san lucas,22.89,-109.91,72.75,56,0,3.00,MX,2026-03-18 23:57:47,comfort rooms,taqueria los paises i
ontario,34.06,-117.65,72.90,42,0,3.44,US,2026-03-19 00:04:56,travelodge ontario,gloria's mexican cuisine
holy rosalia,27.32,-112.28,74.10,45,0,9.66,MX,2026-03-19 00:05:39,sun and sea,the bolerian


## **4.4: Display Restaurant Locations**

In [18]:
mapx.set_tooltip_cols(['hotel_name', 'restaurant_name', 'city', 'country'])

mapx.disp_folium_circles_df(updated_restaurant_df, 'Figure 4.4: Restaurant Locations')

# <br> **Section 5: Tourism Attraction Locations**

## **5.1: Add Tourism Attraction Column to DataFrame**

In [19]:
tourist_attraction_df = updated_restaurant_df.copy()

tourist_attraction_df['tourist_attraction'] = pd.Series(dtype = 'str')

tourist_attraction_df.reset_index(drop = True, inplace = True)


logx.log_write_obj(tourist_attraction_df)

## **5.2: Find Tourism Attraction Locations**

In [20]:
updated_tourist_attraction_df \
    = citipyx.update_location_vacation_df \
        (tourist_attraction_df, 
         'tourist_attraction', 
         'tourism.attraction')

logx.log_write_obj(updated_tourist_attraction_df)

STARTING TOURISM ATTRACTION SEARCH...


Located the following tourism attraction...Steam Train in gundlupet, IN


Located the following tourism attraction...#CABO in gundlupet, IN


Located the following tourism attraction...Conversations With Michael in gundlupet, IN


Located the following tourism attraction...Old mining locomotive in gundlupet, IN


TOURISM ATTRACTION SEARCH COMPLETE...




## **5.3: Display Tourism Attraction Data Set**

In [21]:
pandasx.rtn_fmt_tbl(updated_tourist_attraction_df, 'Table: 5.3: Tourist Attraction Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time,hotel_name,restaurant_name,tourist_attraction
queenstown,-31.90,26.88,70.12,59,1,2.73,ZA,2026-03-18 23:59:45,queens hotel,n6 road house,steam train
cape san lucas,22.89,-109.91,72.75,56,0,3.00,MX,2026-03-18 23:57:47,comfort rooms,taqueria los paises i,#cabo
ontario,34.06,-117.65,72.90,42,0,3.44,US,2026-03-19 00:04:56,travelodge ontario,gloria's mexican cuisine,conversations with michael
holy rosalia,27.32,-112.28,74.10,45,0,9.66,MX,2026-03-19 00:05:39,sun and sea,the bolerian,old mining locomotive


## **5.4: Display Tourism Attraction Locations**

In [22]:
mapx.set_tooltip_cols(['hotel_name', 'restaurant_name', 'tourist_attraction', 'city', 'country'])

mapx.disp_folium_circles_df(updated_tourist_attraction_df, 'Figure 5.4: Tourist Attraction Locations')

In [23]:
# logx.end_program()